# E2B Example: Parallel Anomaly Triage

This notebook creates a realistic incident bundle, sends independent investigation lanes to parallel E2B sandboxes, and asks a coordinator agent to synthesize the likely root cause.

It defaults to `gpt-5.4-mini` because the example relies on multiple parallel worker tool calls and benefits from lower planning latency.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src" / "agents").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "agents").exists():
            repo_root = candidate
            break
        if (candidate / "openai-agents-python-preview" / "src" / "agents").exists():
            repo_root = candidate / "openai-agents-python-preview"
            break

src_root = repo_root / "src"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

env_path = None
for candidate in [Path.cwd(), *Path.cwd().parents, repo_root, *repo_root.parents]:
    maybe_env = candidate / ".env"
    if maybe_env.exists():
        env_path = maybe_env
        break

if env_path is not None:
    try:
        from dotenv import load_dotenv

        load_dotenv(env_path, override=False)
    except Exception:
        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

required_modules = ["agents", "openai", "e2b", "e2b_code_interpreter"]
missing_modules = [
    module_name for module_name in required_modules if importlib.util.find_spec(module_name) is None
]
if missing_modules:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root) + "[e2b]"]
    )

In [ ]:
from agents.extensions.sandbox import E2BSandboxType
from examples.sandbox.extensions.e2b.parallel_anomaly_triage import (
    DEFAULT_INCIDENT_SEED,
    build_incident_bundle,
    require_credentials,
    run_parallel_anomaly_triage,
)

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
SANDBOX_TYPE = E2BSandboxType(os.getenv("E2B_SANDBOX_TYPE", E2BSandboxType.E2B.value))
TEMPLATE = os.getenv("E2B_TEMPLATE") or None
TIMEOUT_SECONDS = int(os.getenv("E2B_TIMEOUT", "900"))
INCIDENT_SEED = int(os.getenv("INCIDENT_SEED", str(DEFAULT_INCIDENT_SEED)))

require_credentials()
incident_bundle = build_incident_bundle(INCIDENT_SEED)
print(incident_bundle["incident_brief.txt"])

In [ ]:
payload = await run_parallel_anomaly_triage(  # type: ignore[top-level-await]  # noqa: F704
    model=MODEL,
    sandbox_type=SANDBOX_TYPE,
    template=TEMPLATE,
    timeout_seconds=TIMEOUT_SECONDS,
    incident_seed=INCIDENT_SEED,
    incident_bundle=incident_bundle,
)
payload["selection"]

In [ ]:
try:
    import pandas as pd

    display(pd.DataFrame(payload["lanes"]))
except Exception:
    print(payload["lanes"])